# V4 Fast — Audio Deepfake Detection (Pre-extracted Features)
## Target: EER < 5% on ASVspoof 2019 LA Eval

**Uses pre-extracted .npy features — no raw audio processing needed!**

Upload your `features_v2` folder as a Kaggle dataset containing:
- `train_X.npy` (25380, 996)
- `train_y.npy` (25380,)
- `dev_X.npy` (24844, 996) 
- `dev_y.npy` (24844,)
- `eval_X.npy` (71237, 996)
- `eval_y.npy` (71237,)

**Key improvements over v3 (10.67% EER):**
1. Deeper LCNN with SE (Squeeze-and-Excitation) attention
2. Feature-level augmentation (Gaussian noise + random masking)
3. Multi-seed ensemble (3 models, score averaging)
4. AdamW with stronger weight decay
5. Extended training with cosine warm restarts
6. Gradient accumulation for smoother updates

In [ ]:
# Cell 1: Imports
import os, warnings, time, copy, gc
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, balanced_accuracy_score
from scipy.optimize import brentq
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Load pre-extracted features
# ─────────────────────────────────────────────────────────
# CHANGE THIS PATH to match your Kaggle dataset location!
# Option A: If you uploaded features_v2 as a Kaggle dataset:
FEAT_DIR = '/kaggle/input/asvspoof-features-v2'
# Option B: If they're inside a larger dataset:
# FEAT_DIR = '/kaggle/input/your-dataset-name/features_v2'
# ─────────────────────────────────────────────────────────

# Feature layout in the 996-dim vectors:
# [0:448]   = FrFT features  (7 angles × 8 chunks × 8 stats)
# [448:704] = WPD features   (32 sub-bands × 8 stats)
# [704:944] = LFCC features  (DISCARDED — caused overfitting)
# [944:996] = Spectral features (DISCARDED)

N_FRFT = 448
N_WPD  = 256
N_FEAT_USED = N_FRFT + N_WPD  # 704

print("Loading pre-extracted features...")
X_train_full = np.load(os.path.join(FEAT_DIR, 'train_X.npy'))
y_train      = np.load(os.path.join(FEAT_DIR, 'train_y.npy'))
X_dev_full   = np.load(os.path.join(FEAT_DIR, 'dev_X.npy'))
y_dev        = np.load(os.path.join(FEAT_DIR, 'dev_y.npy'))
X_eval_full  = np.load(os.path.join(FEAT_DIR, 'eval_X.npy'))
y_eval       = np.load(os.path.join(FEAT_DIR, 'eval_y.npy'))

print(f"Raw shapes: train={X_train_full.shape}, dev={X_dev_full.shape}, eval={X_eval_full.shape}")

# Slice to FrFT + WPD only (first 704 features)
X_train = X_train_full[:, :N_FEAT_USED]
X_dev   = X_dev_full[:, :N_FEAT_USED]
X_eval  = X_eval_full[:, :N_FEAT_USED]

# Free full arrays
del X_train_full, X_dev_full, X_eval_full
gc.collect()

print(f"\nSliced shapes (FrFT + WPD only):")
print(f"  Train: {X_train.shape} | labels: {y_train.shape}")
print(f"  Dev:   {X_dev.shape}   | labels: {y_dev.shape}")
print(f"  Eval:  {X_eval.shape}  | labels: {y_eval.shape}")

for name, y in [('Train', y_train), ('Dev', y_dev), ('Eval', y_eval)]:
    n_real  = int(y.sum())
    n_spoof = len(y) - n_real
    print(f"  {name}: {len(y):,} total | {n_real:,} bonafide | {n_spoof:,} spoof")

In [ ]:
# Cell 3: Configuration

# Training hyperparameters
BATCH        = 256
ACCUM_STEPS  = 2                # effective batch = 512
EPOCHS       = 80               # more epochs
LR           = 3e-4
WEIGHT_DECAY = 1e-3             # stronger L2 with AdamW
SEED_LIST    = [42, 2024, 7]    # 3 seeds for ensemble
PATIENCE     = 15               # early stopping

print(f"Features: {N_FEAT_USED} (FrFT={N_FRFT} + WPD={N_WPD})")
print(f"Training: batch={BATCH}, accum={ACCUM_STEPS}, effective={BATCH*ACCUM_STEPS}")
print(f"Epochs: {EPOCHS}, patience: {PATIENCE}")
print(f"Seeds for ensemble: {SEED_LIST}")
print(f"Optimizer: AdamW (lr={LR}, wd={WEIGHT_DECAY})")

In [ ]:
# Cell 4: Prepare datasets

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_dev_sc   = scaler.transform(X_dev)
X_eval_sc  = scaler.transform(X_eval)


class AudioDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        x = self.X[i].clone()
        if self.augment:
            # Gaussian noise — preserves phase-artifact structure
            if torch.rand(1).item() < 0.3:
                noise_scale = 0.05 if self.y[i].item() == 0 else 0.02
                x = x + torch.randn_like(x) * noise_scale
            # Random feature masking (like SpecAugment for features)
            if torch.rand(1).item() < 0.25:
                mask_len = torch.randint(10, 50, (1,)).item()
                mask_start = torch.randint(0, max(1, len(x) - mask_len), (1,)).item()
                x[mask_start:mask_start + mask_len] = 0
            # Random scaling (subtle)
            if torch.rand(1).item() < 0.2:
                scale = 1.0 + (torch.rand(1).item() - 0.5) * 0.1  # 0.95 to 1.05
                x = x * scale
        return x, self.y[i]


train_ds = AudioDataset(X_train_sc, y_train, augment=True)
dev_ds   = AudioDataset(X_dev_sc,   y_dev,   augment=False)
eval_ds  = AudioDataset(X_eval_sc,  y_eval,  augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
dev_loader   = DataLoader(dev_ds,   batch_size=512, shuffle=False,
                          num_workers=2, pin_memory=True)
eval_loader  = DataLoader(eval_ds,  batch_size=512, shuffle=False,
                          num_workers=2, pin_memory=True)

# Class weights for imbalanced dataset
n_total = len(y_train)
n_real  = int(y_train.sum())
n_spoof = n_total - n_real
class_weight = torch.tensor([
    n_total / (2.0 * n_spoof),   # weight for spoof (class 0)
    n_total / (2.0 * n_real)     # weight for bonafide (class 1)
], dtype=torch.float32).to(device)

print(f"Class weights: spoof={class_weight[0]:.3f}, bonafide={class_weight[1]:.3f}")
print(f"Train batches: {len(train_loader)}")
print("Datasets ready!")

In [ ]:
# Cell 5: Model Architecture — LCNN v4 with SE Attention

class MFM(nn.Module):
    """Max Feature Map: competitive activation that halves channels."""
    def forward(self, x):
        n = x.shape[1] // 2
        return torch.max(x[:, :n], x[:, n:])


class SEBlock(nn.Module):
    """Squeeze-and-Excitation for channel attention."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        se = F.relu(self.fc1(x))
        se = torch.sigmoid(self.fc2(se))
        return x * se


class LCNNBlock(nn.Module):
    """LCNN block: Linear → MFM → BN → SE → Dropout + residual."""
    def __init__(self, in_dim, hidden_dim, dropout=0.3, use_se=True):
        super().__init__()
        out_dim = hidden_dim // 2  # MFM halves
        self.fc  = nn.Linear(in_dim, hidden_dim)
        self.mfm = MFM()
        self.bn  = nn.BatchNorm1d(out_dim)
        self.se  = SEBlock(out_dim) if use_se else nn.Identity()
        self.drop = nn.Dropout(dropout)
        self.residual = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()

    def forward(self, x):
        identity = self.residual(x)
        out = self.drop(self.se(self.bn(self.mfm(self.fc(x)))))
        return out + identity


class LCNN_v4(nn.Module):
    """
    Light CNN v4 — 5-block with SE attention.
    
    Flow: Input(704) → Drop → 1024→512 → 1024→512 → 512→256 → 256→128 → 128→64 → 2
    Each block has: Linear → MFM → BN → SE → Dropout + Residual skip
    """
    def __init__(self, input_dim):
        super().__init__()
        self.input_drop = nn.Dropout(0.15)
        self.block1 = LCNNBlock(input_dim, 1024, dropout=0.5,  use_se=True)
        self.block2 = LCNNBlock(512,       1024, dropout=0.45, use_se=True)
        self.block3 = LCNNBlock(512,        512, dropout=0.4,  use_se=True)
        self.block4 = LCNNBlock(256,        256, dropout=0.3,  use_se=True)
        self.block5 = LCNNBlock(128,        128, dropout=0.2,  use_se=False)
        self.classifier = nn.Linear(64, 2)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.input_drop(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        return self.classifier(x)


model_test = LCNN_v4(N_FEAT_USED)
n_params = sum(p.numel() for p in model_test.parameters())
print(f"LCNN_v4 Architecture:")
print(model_test)
print(f"\nTotal parameters: {n_params:,}")
print(f"Model size: ~{n_params * 4 / 1e6:.1f} MB (float32)")
del model_test

In [ ]:
# Cell 6: Training and evaluation utilities

def compute_eer(labels, scores):
    """EER via interpolation (more accurate than argmin)."""
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    try:
        eer = brentq(lambda x: interp1d(fpr, fnr)(x) - x, 0, 1)
        thresh = float(interp1d(fpr, thresholds)(eer))
    except:
        idx = np.nanargmin(np.abs(fnr - fpr))
        eer = float(np.mean([fpr[idx], fnr[idx]]))
        thresh = float(thresholds[idx])
    return eer * 100, thresh


def evaluate(mdl, loader, return_scores=False):
    mdl.eval()
    all_labels, all_scores = [], []
    correct = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = mdl(X)
            probs  = torch.softmax(logits, dim=1)[:, 1]
            correct += (logits.argmax(1) == y).sum().item()
            all_labels.extend(y.cpu().numpy())
            all_scores.extend(probs.cpu().numpy())
    acc = 100.0 * correct / len(all_labels)
    labels = np.array(all_labels)
    scores = np.array(all_scores)
    eer, thresh = compute_eer(labels, scores)
    bal_acc = balanced_accuracy_score(labels, (scores >= thresh).astype(int)) * 100
    if return_scores:
        return eer, acc, bal_acc, scores, labels
    return eer, acc, bal_acc


def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

print("Training utilities ready!")

In [ ]:
# Cell 7: Core training function

def train_model(seed, tag, epochs=EPOCHS, patience=PATIENCE):
    """Train one LCNN_v4 model. Returns model, history, eval results."""
    set_seed(seed)
    print(f"\n{'='*70}")
    print(f"  Training {tag} | seed={seed} | max_epochs={epochs}")
    print(f"{'='*70}")

    model = LCNN_v4(input_dim=N_FEAT_USED).to(device)

    criterion = nn.CrossEntropyLoss(
        weight=class_weight,
        label_smoothing=0.1
    )
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR,
        weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999)
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )

    history = {'loss': [], 'dev_eer': [], 'dev_acc': [], 'dev_bal': []}
    best_dev_eer = float('inf')
    best_epoch = 0
    patience_ctr = 0
    best_state = None
    save_path = f'/kaggle/working/best_{tag}.pt'

    header = f"{'Ep':>4} | {'Loss':>8} | {'Dev EER':>9} | {'Dev Acc':>8} | {'Bal':>7} | {'LR':>10} | {'Time':>5}"
    print(header)
    print("─" * len(header))

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        t0 = time.time()
        optimizer.zero_grad()

        for bi, (X, y) in enumerate(train_loader):
            X, y = X.to(device), y.to(device)
            loss = criterion(model(X), y) / ACCUM_STEPS
            loss.backward()

            if (bi + 1) % ACCUM_STEPS == 0 or (bi + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=3.0)
                optimizer.step()
                optimizer.zero_grad()

            total_loss += loss.item() * ACCUM_STEPS

        scheduler.step()
        avg_loss = total_loss / len(train_loader)
        dev_eer, dev_acc, dev_bal = evaluate(model, dev_loader)
        elapsed = time.time() - t0
        lr_now = optimizer.param_groups[0]['lr']

        history['loss'].append(avg_loss)
        history['dev_eer'].append(dev_eer)
        history['dev_acc'].append(dev_acc)
        history['dev_bal'].append(dev_bal)

        flag = ''
        if dev_eer < best_dev_eer:
            best_dev_eer = dev_eer
            best_epoch = ep
            patience_ctr = 0
            flag = ' ← best'
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, save_path)
        else:
            patience_ctr += 1

        print(f"{ep:>4} | {avg_loss:>8.4f} | {dev_eer:>8.2f}% | {dev_acc:>7.2f}% | {dev_bal:>6.2f}% | {lr_now:>10.2e} | {elapsed:>4.1f}s{flag}")

        if patience_ctr >= patience:
            print(f"\n⚡ Early stopping at epoch {ep}")
            break

    # Load best checkpoint and evaluate on eval set
    model.load_state_dict(best_state)
    eval_eer, eval_acc, eval_bal, eval_scores, eval_labels = evaluate(
        model, eval_loader, return_scores=True
    )

    print(f"\n✓ Best Dev EER: {best_dev_eer:.2f}% (epoch {best_epoch})")
    print(f"✓ Eval EER: {eval_eer:.2f}% | Acc: {eval_acc:.2f}% | Bal: {eval_bal:.2f}%")

    return model, history, eval_eer, eval_acc, eval_bal, eval_scores, eval_labels

print("Training function ready!")

In [ ]:
# Cell 8: Train 3-model ensemble

results = []
all_eval_scores = []
all_dev_scores = []
eval_labels_saved = None
dev_labels_saved = None

for seed in SEED_LIST:
    tag = f"v4_s{seed}"
    model, hist, e_eer, e_acc, e_bal, e_scores, e_labels = train_model(seed, tag)

    results.append({
        'seed': seed, 'tag': tag,
        'eval_eer': e_eer, 'eval_acc': e_acc, 'eval_bal': e_bal,
        'history': hist
    })
    all_eval_scores.append(e_scores)
    eval_labels_saved = e_labels

    # Get dev scores too
    _, _, _, d_scores, d_labels = evaluate(model, dev_loader, return_scores=True)
    all_dev_scores.append(d_scores)
    dev_labels_saved = d_labels

    del model
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n{'='*60}")
print("  Individual Model Results")
print(f"{'='*60}")
for r in results:
    print(f"  Seed {r['seed']:>5}: EER={r['eval_eer']:.2f}%  Acc={r['eval_acc']:.2f}%  Bal={r['eval_bal']:.2f}%")

In [ ]:
# Cell 9: Ensemble fusion and final evaluation

# Score averaging
ens_eval = np.mean(all_eval_scores, axis=0)
ens_dev  = np.mean(all_dev_scores, axis=0)

# EER on eval
ens_eer, ens_thresh = compute_eer(eval_labels_saved, ens_eval)
ens_preds = (ens_eval >= ens_thresh).astype(int)
ens_acc = 100.0 * np.mean(ens_preds == eval_labels_saved)
ens_bal = balanced_accuracy_score(eval_labels_saved, ens_preds) * 100

# Dev-calibrated threshold
dev_eer, dev_thresh = compute_eer(dev_labels_saved, ens_dev)
cal_preds = (ens_eval >= dev_thresh).astype(int)
cal_acc = 100.0 * np.mean(cal_preds == eval_labels_saved)
cal_bal = balanced_accuracy_score(eval_labels_saved, cal_preds) * 100

print("="*65)
print("  ENSEMBLE RESULTS")
print("="*65)
print(f"  Models: {len(SEED_LIST)} × LCNN_v4, score averaging")
print(f"")
print(f"  Eval EER       : {ens_eer:.2f}%")
print(f"  Eval Accuracy  : {ens_acc:.2f}%")
print(f"  Eval Bal Acc   : {ens_bal:.2f}%")
print(f"")
print(f"  Dev-calibrated threshold ({dev_thresh:.4f}):")
print(f"    Accuracy : {cal_acc:.2f}%")
print(f"    Bal Acc  : {cal_bal:.2f}%")
print(f"")
print(f"  VERSION COMPARISON:")
print(f"    v1 (448 feats, original)        : 11.79% EER")
print(f"    v2 (996 feats, overfit)          : 13.55% EER")
print(f"    v3 (704 feats, single model)     : 10.67% EER")
for r in results:
    print(f"    v4 seed {r['seed']} (individual)       : {r['eval_eer']:.2f}% EER")
print(f"    v4 ENSEMBLE ({len(SEED_LIST)} models)          : {ens_eer:.2f}% EER  ← NEW BEST")
print(f"")
improvement = 10.67 - ens_eer
print(f"  Improvement over v3: {improvement:.2f}% absolute ({improvement/10.67*100:.1f}% relative)")
print("="*65)

In [ ]:
# Cell 10: Visualization

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle(f'V4 Ensemble — Eval EER = {ens_eer:.2f}%', fontsize=14, fontweight='bold')

colors = ['#2196F3', '#4CAF50', '#FF9800']

# Row 1: Training curves for each seed
for i, r in enumerate(results):
    h = r['history']
    ep = range(1, len(h['loss']) + 1)
    c = colors[i]
    axes[0][0].plot(ep, h['loss'], color=c, lw=1.5, label=f"Seed {r['seed']}")
    axes[0][1].plot(ep, h['dev_eer'], color=c, lw=1.5, label=f"Seed {r['seed']}")
    axes[0][2].plot(ep, h['dev_acc'], color=c, lw=1.5, label=f"Seed {r['seed']}")

axes[0][0].set_title('Training Loss'); axes[0][0].set_ylabel('Loss')
axes[0][0].legend(); axes[0][0].grid(True, alpha=0.3)

axes[0][1].set_title('Dev EER (%)')
axes[0][1].axhline(ens_eer, color='red', ls='--', lw=2, label=f'Ensemble Eval={ens_eer:.2f}%')
axes[0][1].legend(); axes[0][1].grid(True, alpha=0.3)

axes[0][2].set_title('Dev Accuracy (%)')
axes[0][2].legend(); axes[0][2].grid(True, alpha=0.3)

# Row 2: Comparison, score distribution, ROC
versions = ['v1', 'v2', 'v3']
eers_old = [11.79, 13.55, 10.67]
for r in results:
    versions.append(f"v4\ns{r['seed']}")
    eers_old.append(r['eval_eer'])
versions.append('v4\nEnsemble')
eers_old.append(ens_eer)

bar_colors = ['#90CAF9', '#EF9A9A', '#A5D6A7'] + colors[:len(results)] + ['#4CAF50']
bars = axes[1][0].bar(versions, eers_old, color=bar_colors, edgecolor='white', width=0.55)
for bar, eer in zip(bars, eers_old):
    axes[1][0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                    f'{eer:.2f}%', ha='center', fontsize=8, fontweight='bold')
axes[1][0].set_ylabel('EER (%) — lower is better')
axes[1][0].set_title('Version Comparison')
axes[1][0].set_ylim(0, max(eers_old)*1.25)
axes[1][0].grid(axis='y', alpha=0.3)

# Score distribution
axes[1][1].hist(ens_eval[eval_labels_saved==1], bins=100, alpha=0.7,
                color='steelblue', label='Bonafide', density=True)
axes[1][1].hist(ens_eval[eval_labels_saved==0], bins=100, alpha=0.7,
                color='tomato', label='Spoof', density=True)
axes[1][1].axvline(ens_thresh, color='green', ls='--', lw=2,
                   label=f'EER thresh={ens_thresh:.3f}')
axes[1][1].set_title('Ensemble Score Distribution')
axes[1][1].set_xlabel('Bonafide Probability')
axes[1][1].legend(); axes[1][1].grid(True, alpha=0.3)

# ROC
fpr, tpr, _ = roc_curve(eval_labels_saved, ens_eval, pos_label=1)
from sklearn.metrics import auc
roc_auc = auc(fpr, tpr)
axes[1][2].plot(fpr, tpr, 'b-', lw=2, label=f'v4 Ensemble (AUC={roc_auc:.4f})')
axes[1][2].plot([0,1],[0,1], 'k--', alpha=0.3)
axes[1][2].set_xlabel('FPR'); axes[1][2].set_ylabel('TPR')
axes[1][2].set_title('ROC Curve')
axes[1][2].legend(); axes[1][2].grid(True, alpha=0.3)

for ax in axes.flat:
    if not ax.get_xlabel():
        ax.set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('/kaggle/working/v4_results.png', dpi=150)
plt.show()
print("Saved → v4_results.png")

In [ ]:
# Cell 11: Final summary

n_params = sum(p.numel() for p in LCNN_v4(N_FEAT_USED).parameters())

print("═"*65)
print("  FINAL RESULTS — V4 AUDIO DEEPFAKE DETECTION")
print("═"*65)
print(f"")
print(f"  Dataset: ASVspoof 2019 LA")
print(f"  Train: {len(y_train):,} | Dev: {len(y_dev):,} | Eval: {len(y_eval):,}")
print(f"")
print(f"  Features: {N_FEAT_USED} (FrFT={N_FRFT} + WPD={N_WPD})")
print(f"  Model: LCNN_v4 with SE attention ({n_params:,} params each)")
print(f"  Ensemble: {len(SEED_LIST)} models, score averaging")
print(f"")
print(f"  ┌────────────────────────────────────────────┐")
print(f"  │  Eval EER      : {ens_eer:>6.2f}%               │")
print(f"  │  Eval Accuracy : {ens_acc:>6.2f}%               │")
print(f"  │  Eval Bal Acc  : {ens_bal:>6.2f}%               │")
print(f"  │  ROC AUC       : {roc_auc:>6.4f}                │")
print(f"  └────────────────────────────────────────────┘")
print(f"")
print(f"  Saved files:")
for seed in SEED_LIST:
    print(f"    best_v4_s{seed}.pt")
print(f"    v4_results.png")
print(f"")
print(f"  Improvement: v3 (10.67%) → v4 ({ens_eer:.2f}%) = {10.67-ens_eer:.2f}% reduction")
print("═"*65)